In [8]:
import pdfplumber

conversations = []

with pdfplumber.open("data/parliament.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            conversations.append(text)

full_text = "\n".join(conversations)

# save to txt
with open("data/parliament.txt", "w") as f:
    f.write(full_text)

In [42]:
# Section headers that are navigational, not real topics
SKIP_TOPICS = {
    "THE HANSARD", "QUESTIONS AND STATEMENTS", "REQUESTS FOR STATEMENTS",
    "REQUEST FOR STATEMENT", "COMMUNICATION FROM THE CHAIR", "STATEMENTS",
    "MOTIONS", "PRAYERS", "NEXT ORDER", "BUSINESS FOR THE WEEK",
    "PAPERS", "BILLS", "NOTICES OF MOTION", "ADJOURNMENT",
    "NOTING OF REPORT OF KENYA",
    "QUESTIONS AND STATEMENTS REQUEST FOR STATEMENT",
    "QUESTIONS AND STATEMENTS REQUESTS FOR STATEMENTS",
}

In [23]:
import re
import json

def extract_date(text):
    match = re.search(
        r'\b(\d{1,2}(?:st|nd|rd|th)\s+\w+\s+\d{4})\b', text
    )
    return match.group(1) if match else "Unknown Date"

def extract_parliament_info(text):
    # Matches "THIRTEENTH PARLIAMENT" etc.
    parliament = re.search(r'((?:FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH|'
                           r'EIGHTH|NINTH|TENTH|ELEVENTH|TWELFTH|THIRTEENTH|'
                           r'FOURTEENTH|FIFTEENTH)\s+PARLIAMENT)', text)
    
    # Matches "NATIONAL ASSEMBLY" or "SENATE"
    chamber = re.search(r'(NATIONAL ASSEMBLY|SENATE)', text)
    
    return {
        "parliament": parliament.group(1) if parliament else "Unknown Parliament",
        "chamber": chamber.group(1) if chamber else "Unknown Chamber"
    }

In [43]:
def preprocess(text):
    """Remove all noise before any parsing happens."""
    # Disclaimer blocks
    text = re.sub(r'Disclaimer:.*?Hansard Editor\.', '', text, flags=re.DOTALL)
    # Page running headers e.g. "30th April 2026 National Assembly Debates 12"
    text = re.sub(r'\d{1,2}\w{2}\s+\w+\s+\d{4}\s+National Assembly Debates\s+\d+', '', text)
    # Stage directions in brackets/parentheses
    text = re.sub(r'\[.*?\]', '', text, flags=re.DOTALL)
    text = re.sub(r'\((?!Hon\.|The Temp|The Dep)[^)]{0,80}\)', '', text)
    # Closing salutations — "Thank you, Hon. Speaker." or "Thank you, Hon. Deputy Speaker."
    # These appear BEFORE the next real speaker label and contaminate it
    text = re.sub(
        r'Thank you,?\s*\n+Hon\.\s+(?:Deputy\s+)?Speaker\.?\s*\n',
        '\n', text
    )
    # Standalone "Hon. Speaker." lines (closing phrases, not actual speakers)
    text = re.sub(r'\nHon\.\s+(?:Temporary\s+|Deputy\s+)?Speaker\.\s*\n', '\n', text)
    return text

In [46]:
def extract_clean_speaker(raw):
    """
    Given a raw speaker string (possibly with preamble noise),
    extract only the final clean speaker label.
    """
    # Find all Hon./Speaker labels in the raw string
    candidates = re.findall(
        r'((?:The\s+)?(?:Temporary Speaker|Deputy Speaker|Hon\.)\s+[\w\s\.\(\),]+)',
        raw
    )
    if candidates:
        # Last one is the actual speaker; strip trailing noise
        label = candidates[-1].strip().rstrip('.,')
        # Keep only the first line (remove any trailing preamble)
        label = label.split('\n')[0].strip()
        return label
    return raw.split('\n')[0].strip()

In [47]:
def classify_role(speaker):
    """Infer the speaker's role from their label."""
    s = speaker.lower()
    if "temporary speaker" in s or "deputy speaker" in s or "speaker" in s:
        return "presiding_officer"
    if "nominated" in s:
        return "nominated_member"
    # Extract party affiliation if present e.g. "(Rabai, PAA)"
    party_match = re.search(r'\([\w\s]+,\s*([\w]+)\)', speaker)
    party = party_match.group(1).lower() if party_match else "unknown"
    return f"elected_member_{party}"

In [30]:
def extract_constituency(speaker):
    """Pull out constituency/county from speaker label."""
    match = re.search(r'\(([\w\s]+),\s*[\w]+\)', speaker)
    return match.group(1).strip() if match else None

In [48]:
# Matches a clean speaker label: Hon. Name (Constituency, Party)
# or The Temporary Speaker (Hon. Name)
SPEAKER_PATTERN = re.compile(
    r'((?:The\s+)?(?:Hon\.|Temporary Speaker|Deputy Speaker)'
    r'[\w\s\.\(\),\-]*?'
    r'(?:\([\w\s,]+\))?)'
    r'\s*:',
    re.MULTILINE
)


In [49]:
def parse_turns(text):
    """Split text into (speaker, utterance) pairs, cleaning speaker labels."""
    # Find all speaker positions
    matches = list(SPEAKER_PATTERN.finditer(text))
    if not matches:
        return []

    turns = []
    for i, match in enumerate(matches):
        raw_speaker = match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        utterance = text[start:end].strip()

        if len(utterance) < 40:
            continue  # skip filler

        speaker = extract_clean_speaker(raw_speaker)
        turns.append((speaker, utterance))

    return turns

In [24]:
def is_skip_topic(topic):
    # Normalize and check if it's a generic section header
    normalized = topic.strip().upper()
    for skip in SKIP_TOPICS:
        if normalized == skip or normalized.startswith(skip + "\n"):
            return True
    return False


In [44]:
def clean_topic(raw_topic):
    lines = [l.strip() for l in raw_topic.strip().splitlines() if l.strip()]
    skip_normalized = {" ".join(s.upper().split()) for s in SKIP_TOPICS}
    filtered = [l for l in lines if " ".join(l.upper().split()) not in skip_normalized]
    return " ".join(filtered) if filtered else " ".join(lines)

In [ ]:
def is_valid_topic(topic):
    normalized = " ".join(topic.upper().split())  # collapse whitespace
    if normalized in SKIP_TOPICS:
        return False
    # Also skip if it starts with any skip topic
    for skip in SKIP_TOPICS:
        if normalized.startswith(skip):
            return False
    if len(normalized.replace(" ", "")) < 10:
        return False
    return True


In [45]:
# Matches a clean speaker label: Hon. Name (Constituency, Party)
# or The Temporary Speaker (Hon. Name)
SPEAKER_PATTERN = re.compile(
    r'((?:The\s+)?(?:Hon\.|Temporary Speaker|Deputy Speaker)'
    r'[\w\s\.\(\),\-]*?'
    r'(?:\([\w\s,]+\))?)'
    r'\s*:',
    re.MULTILINE
)

In [50]:
def parse_hansard_by_speaker(text):
    chunks = []
    date = extract_date(text)
    parliament_info = extract_parliament_info(text)

    sections = re.split(r'\n((?:[A-Z][A-Z\s,\-]+\n?){1,4})\n', text)

    current_topic = None
    for section in sections:
        stripped = section.strip()

        # Detect heading
        if re.match(r'^[A-Z][A-Z\s,\-\n]{8,}$', stripped) and len(stripped) < 300:
            current_topic = stripped
            continue

        if not current_topic:
            continue

        topic_label = clean_topic(current_topic)
        if not is_valid_topic(topic_label):
            current_topic = None
            continue

        turns = parse_turns(stripped)
        if not turns:
            current_topic = None
            continue

        # Chunk 1: Full topic conversation
        full_text = f"Topic: {topic_label}\n\n"
        all_speakers = []
        for speaker, utterance in turns:
            full_text += f"{speaker}: {utterance}\n\n"
            all_speakers.append(speaker)

        chunks.append({
            "text": full_text.strip(),
            "chunk_type": "full_topic",
            "metadata": {
                "topic": topic_label,
                "speakers": list(set(all_speakers)),
                "date": date,
                "parliament": parliament_info["parliament"],
                "chamber": parliament_info["chamber"],
                "source": "Kenya Hansard"
            }
        })

        # Chunk 2: Individual speaker turns
        for speaker, utterance in turns:
            constituency = extract_constituency(speaker)
            role = classify_role(speaker)

            chunks.append({
                "text": f"Topic: {topic_label}\n{speaker}: {utterance}",
                "chunk_type": "speaker_turn",
                "metadata": {
                    "topic": topic_label,
                    "speaker": speaker,
                    "constituency": constituency,
                    "role": role,
                    "date": date,
                    "parliament": parliament_info["parliament"],
                    "chamber": parliament_info["chamber"],
                    "source": "Kenya Hansard"
                }
            })

        current_topic = None

    return chunks

In [51]:
def save_chunks(chunks, output_path="data/hansard_rag_chunks.jsonl"):
    with open(output_path, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    full = sum(1 for c in chunks if c["chunk_type"] == "full_topic")
    speaker = sum(1 for c in chunks if c["chunk_type"] == "speaker_turn")
    print(f"Saved {len(chunks)} chunks ({full} topic, {speaker} speaker turns)")

In [52]:
chunks = parse_hansard_by_speaker(full_text)
save_chunks(chunks)

Saved 120 chunks (17 topic, 103 speaker turns)
